# Установка библиотек

In [371]:
# %pip install pandas numpy plotly nbformat scikit-learn

# Импорты

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import abc as abc
import sklearn as skl

# Самописная Линеная Агрессия

In [373]:
class Error(abc.ABC):
    @abc.abstractmethod
    def calculate(self, y_true, y_pred):
        pass


class MSE(Error):

    def calculate(self, y_true, y_pred):
        return np.mean((y_true - y_pred) ** 2)
    
class MAE(Error):
    
    def calculate(self, y_true, y_pred):
        return np.mean(np.abs(y_true - y_pred))

In [ ]:
class CustomLinReg(abc.ABC):

    def __init__(
            self, 
            target: np.ndarray , 
            features: np.ndarray
        ):
        self.target = target
        self.features = features
        self.weights = None
        self.bias = None

    @abc.abstractmethod
    def fit(self , 
            target_error_value :  np.float32,  
            error: Error , 
            max_iter: np.int32 = 10_000,
            learning_rate: np.float32 = 0.01
            ) -> None:
        pass

    @abc.abstractmethod
    def predict(self, features: np.ndarray) -> np.ndarray:
        pass



In [375]:

# Самописная Линейная Регрессия с аналитическим решением
#
# w = (X^T * X)^(-1) * X^T * y
# b = mean(y) - mean(X) * w
class CustomAnalLinReg(CustomLinReg):

    def fit(self , 
            target_error_value :  np.float32,   
            error: Error,
            max_iter: np.int32 = 10_000,
            learning_rate: np.float32 = 0.01
        ) -> None:

        self.weights = np.linalg.inv(self.features.T @ self.features) @ self.features.T @ self.target
        self.bias = np.mean(self.target) - np.mean(self.features, axis=0) @ self.weights

    def predict(self, features):
        
        if self.weights is None or self.bias is None:
            raise ValueError("Model is not fitted yet.")
        
        return features @ self.weights + self.bias

In [376]:

# Самописная Линейная Регрессия с градиентным спуском
#
# w = w - lr * dL/dw
# b = b - lr * dL/db
#
class CustomComputeLinReg(CustomLinReg):

    def fit(self , 
            target_error_value :  np.float32,  
            error: Error , 
            max_iter: np.int32 = 10_000,
            learning_rate: np.float32 = 0.01
        ) -> None:

        self.weights = np.random.randn(self.features.shape[1])
        self.bias = np.random.randn()

        for _ in range(max_iter):
            
            predicitions = self.predict(self.features)
            error_value = error.calculate(self.target, predicitions)

            if error_value <= target_error_value:
                break

            dw = (-2 / self.features.shape[0]) * (self.features.T @ (self.target - predicitions))
            db = (-2 / self.features.shape[0]) * np.sum(self.target - predicitions)

            self.weights -= learning_rate * dw
            self.bias -= learning_rate * db        

    def predict(self, features):
        
        if self.weights is None or self.bias is None:
            raise ValueError("Model is not fitted yet.")

        return features @ self.weights + self.bias

# Самописные деревья

# Дата сет
[Ссылка на kaggle](https://www.kaggle.com/datasets/ruiqurm/lianjia)

## Описание дата сета

* **`url`**: URL, по которому получены данные.

* **`id`**: ID сделки.

* **`Lng` и `Lat`**: координаты (долгота и широта), использующие протокол BD09.

* **`Cid`**: ID жилого комплекса.

* **`tradeTime`**: дата сделки.

* **`DOM`**: количество дней, которые объект находился на рынке (подробнее: [Days on market — Википедия](https://en.wikipedia.org/wiki/Days_on_market)).

* **`followers`**: количество человек, которые следили за объектом.

* **`totalPrice`**: общая стоимость недвижимости.

* **`price`**: средняя стоимость за квадратный метр.

* **`square`**: площадь объекта (в м²).

* **`livingRoom`**: количество жилых комнат.

* **`drawingRoom`**: количество гостиных.

* **`kitchen`**: количество кухонь.

* **`bathroom`**: количество ванных комнат.

* **`floor`**: этаж или высота здания (в следующей версии данные на китайском будут переведены на английский).

* **`buildingType`**: тип здания:

  * `1` — башня
  * `2` — бунгало
  * `3` — комбинация башни и плиты
  * `4` — плита

* **`constructionTime`**: год постройки.

* **`renovationCondition`**: состояние ремонта:

  * `1` — другое
  * `2` — черновая отделка
  * `3` — простая отделка
  * `4` — качественная отделка

* **`buildingStructure`**: тип конструкции здания:

  * `1` — неизвестно
  * `2` — смешанная
  * `3` — кирпич и дерево
  * `4` — кирпич и бетон
  * `5` — сталь
  * `6` — сталь-бетонная композиция

* **`ladderRatio`**: соотношение между количеством жителей на этаже и количеством лифтов — показывает, сколько лифтов приходится в среднем на одного жителя.

* **`elevator`**: наличие лифта:

  * `1` — есть лифт
  * `0` — нет лифта

* **`fiveYearsProperty`**: указывает, владел ли собственник недвижимостью менее 5 лет.



In [377]:
df = pd.read_csv('Housing price in Beijing.csv', encoding='gbk')

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 318851 entries, 0 to 318850
Data columns (total 26 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   url                  318851 non-null  object 
 1   id                   318851 non-null  object 
 2   Lng                  318851 non-null  float64
 3   Lat                  318851 non-null  float64
 4   Cid                  318851 non-null  int64  
 5   tradeTime            318851 non-null  object 
 6   DOM                  160874 non-null  float64
 7   followers            318851 non-null  int64  
 8   totalPrice           318851 non-null  float64
 9   price                318851 non-null  int64  
 10  square               318851 non-null  float64
 11  livingRoom           318851 non-null  object 
 12  drawingRoom          318851 non-null  object 
 13  kitchen              318851 non-null  int64  
 14  bathRoom             318851 non-null  object 
 15  floor            

/var/folders/y7/3khx9vbx74z9_0bd7nswgf080000gn/T/ipykernel_2284/3510608711.py:1: DtypeWarning:

Columns (1,11,12,14) have mixed types. Specify dtype option on import or set low_memory=False.



In [391]:
df_normalized.describe()

,Lng,Lat,Cid,DOM,followers,totalPrice,price,square,livingRoom,drawingRoom,...,buildingType,constructionTime,renovationCondition,buildingStructure,ladderRatio,elevator,fiveYearsProperty,subway,district,communityAverage
count,30198.000000,30198.000000,3.019800e+04,30198.000000,30198.000000,30198.000000,30198.000000,30198.000000,30198.000000,30198.000000,...,30198.000000,30198.000000,30198.000000,30198.000000,30198.000000,30198.000000,30198.000000,30198.000000,30198.000000,30198.000000
mean,116.422031,39.959270,1.176604e+12,36.264587,28.232896,1.011957,48200.974634,1.026376,2.481224,1.421783,...,2.887178,2003.840850,3.227896,5.038148,0.424313,0.736175,0.623286,0.550666,6.477714,57950.715975
std,0.109874,0.098269,2.763090e+12,53.498256,41.368954,0.682240,12024.429624,0.935659,0.657865,0.512026,...,1.279558,5.670493,1.065706,1.681340,0.170258,0.440713,0.484570,0.497435,2.530043,16152.406572
min,116.072514,39.627030,1.111027e+12,1.000000,0.000000,0.002443,13621.000000,0.000276,0.000000,0.000000,...,1.000000,1950.000000,1.000000,2.000000,0.042000,0.000000,0.000000,0.000000,1.000000,24941.000000
25%,116.343244,39.893778,1.111027e+12,1.000000,4.000000,0.409423,40069.500000,0.349305,2.000000,1.000000,...,1.000000,2001.000000,3.000000,4.000000,0.300000,0.000000,0.000000,0.000000,6.000000,45854.000000
50%,116.422761,39.939588,1.111027e+12,15.000000,15.000000,0.926707,47216.500000,0.731544,2.000000,1.000000,...,3.000000,2004.000000,4.000000,6.000000,0.500000,1.000000,1.000000,1.000000,7.000000,56080.000000
75%,116.488446,40.049793,1.111027e+12,51.000000,35.000000,1.542884,55676.250000,1.487054,3.000000,2.000000,...,4.000000,2007.000000,4.000000,6.000000,0.500000,1.000000,1.000000,1.000000,7.000000,65820.000000
max,116.711337,40.235619,1.184867e+14,1464.000000,795.000000,2.463345,87109.000000,8.988026,7.000000,5.000000,...,4.000000,2016.000000,4.000000,6.000000,3.000000,1.000000,1.000000,1.000000,13.000000,165490.000000


## Преобразование типов и базовая предобработка

In [ ]:
    
df['livingRoom'] = pd.to_numeric(df['livingRoom'], errors='coerce')
df['drawingRoom'] = pd.to_numeric(df['drawingRoom'], errors='coerce')
df['kitchen'] = pd.to_numeric(df['kitchen'], errors='coerce')
df['bathRoom'] = pd.to_numeric(df['bathRoom'], errors='coerce')
df['constructionTime'] = pd.to_numeric(df['constructionTime'], errors='coerce')




## Выбираем признаки и целевую переменную

In [380]:
TARGET_NAME = 'totalPrice'
FEATURES_NAMES = ['square']

## Чистим дата сет

In [383]:
df_cleaned = df.dropna(subset=[TARGET_NAME] + FEATURES_NAMES).copy()

df_cleaned = df_cleaned.where(df_cleaned['square'] > 10).dropna()
df_cleaned = df_cleaned.where(df_cleaned['totalPrice'] > 0).dropna()


q1_totalPrice = df_cleaned['totalPrice'].quantile(0.1)
q3_totalPrice = df_cleaned['totalPrice'].quantile(0.9)
df_cleaned = df_cleaned.where(df_cleaned['totalPrice'] >= q1_totalPrice).where(df_cleaned['totalPrice'] <= q3_totalPrice).dropna()


df_normalized = df_cleaned.copy()

features_mean = df_normalized[FEATURES_NAMES].mean()
features_std = df_normalized[FEATURES_NAMES].std()

df_normalized[FEATURES_NAMES] = (df_normalized[FEATURES_NAMES] - features_mean) / features_std

target_mean = df_normalized[TARGET_NAME].mean()
target_std = df_normalized[TARGET_NAME].std()
df_normalized[TARGET_NAME] = (df_normalized[TARGET_NAME] - target_mean) / target_std


df_normalized = df_normalized.where(df_normalized['square'] > 0).dropna()
df_normalized = df_normalized.where(df_normalized['totalPrice'] > 0).dropna()


print(f"Используемая целевая переменная: {TARGET_NAME}")
print(f"Используемые признаки: {FEATURES_NAMES}")
print(f"Размер очищенного и нормализованного датасета: {df_normalized.shape}")
print(f"Среднее значение целевой переменной: {target_mean}")
print(f"Стандартное отклонение целевой переменной: {target_std}")

df_normalized.info()

Используемая целевая переменная: totalPrice
Используемые признаки: ['square']
Размер очищенного и нормализованного датасета: (30198, 26)
Среднее значение целевой переменной: 376.1788961649457
Стандартное отклонение целевой переменной: 131.45586407447323
<class 'pandas.core.frame.DataFrame'>
Index: 30198 entries, 0 to 318841
Data columns (total 26 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   url                  30198 non-null  object 
 1   id                   30198 non-null  object 
 2   Lng                  30198 non-null  float64
 3   Lat                  30198 non-null  float64
 4   Cid                  30198 non-null  float64
 5   tradeTime            30198 non-null  object 
 6   DOM                  30198 non-null  float64
 7   followers            30198 non-null  float64
 8   totalPrice           30198 non-null  float64
 9   price                30198 non-null  float64
 10  square               30198 non-null 

In [ ]:
df_normalized.describe()

,Lng,Lat,Cid,DOM,followers,totalPrice,price,square,livingRoom,drawingRoom,...,buildingType,constructionTime,renovationCondition,buildingStructure,ladderRatio,elevator,fiveYearsProperty,subway,district,communityAverage
count,36422.000000,36422.000000,3.642200e+04,36422.000000,36422.000000,36422.000000,36422.000000,36422.000000,36422.000000,36422.000000,...,36422.000000,36422.000000,36422.000000,36422.000000,36422.000000,36422.000000,36422.000000,36422.000000,36422.000000,36422.000000
mean,116.419643,39.955614,1.162197e+12,42.365686,29.121163,1.124822,57738.341689,1.117247,2.629153,1.505628,...,2.850585,2004.025671,3.287601,5.295975,0.451933,0.802784,0.596096,0.601340,6.619873,66373.083823
std,0.099738,0.085745,2.440841e+12,58.938982,41.070729,1.230356,19549.515088,1.076229,0.714090,0.527031,...,1.275237,5.525167,1.048642,1.497931,0.199335,0.397902,0.490685,0.489629,2.510406,20999.744302
min,116.081646,39.627030,1.111027e+12,1.000000,0.000000,0.000312,13621.000000,0.000064,0.000000,0.000000,...,1.000000,1950.000000,1.000000,2.000000,0.022000,0.000000,0.000000,0.000000,1.000000,24941.000000
25%,116.346630,39.898397,1.111027e+12,1.000000,5.000000,0.315747,44539.500000,0.343023,2.000000,1.000000,...,1.000000,2001.000000,3.000000,6.000000,0.333000,1.000000,0.000000,0.000000,6.000000,50908.000000
50%,116.424904,39.942343,1.111027e+12,20.000000,16.000000,0.745527,53710.500000,0.833749,3.000000,1.000000,...,3.000000,2004.000000,4.000000,6.000000,0.500000,1.000000,1.000000,1.000000,7.000000,62598.000000
75%,116.481431,40.014025,1.111027e+12,61.000000,37.000000,1.523763,67238.000000,1.549002,3.000000,2.000000,...,4.000000,2008.000000,4.000000,6.000000,0.500000,1.000000,1.000000,1.000000,8.000000,78260.000000
max,116.711337,40.235619,1.184867e+14,1464.000000,795.000000,17.700137,150000.000000,15.284111,7.000000,5.000000,...,4.000000,2016.000000,4.000000,6.000000,3.000000,1.000000,1.000000,1.000000,13.000000,183109.000000


In [ ]:
df_cut = df_normalized[:10_000]

# Сравнения

## Исходные линреги

In [ ]:
target_data = df_cut[TARGET_NAME].to_numpy()
features_data = df_cut[FEATURES_NAMES].to_numpy()




# # Инициализация моделей
anal_linreg  : CustomLinReg = CustomAnalLinReg(
    target=target_data,
    features=features_data
)
comp_linreg : CustomLinReg  = CustomComputeLinReg(
    target=target_data,
    features=features_data
)

anal_linreg.fit(
    target_error_value=0.1, 
    error=MSE()
) 
comp_linreg.fit(
    target_error_value=0.1, 
    error=MSE(),     
    max_iter=1000,
    learning_rate=0.01
)

# print(target_data.shape, features_data.shape)

w_cap = np.linalg.inv(features_data.T @ features_data) @ features_data.T @ target_data
weights = w_cap
bias = np.mean(target_data) - np.mean(features_data, axis=0) @ weights


## Ошибки

In [390]:
y_pred_anal = anal_linreg.predict(features_data)
y_pred_comp = comp_linreg.predict(features_data)

# Расчет ошибки
mse_anal = MSE().calculate(target_data, y_pred_anal)
mse_comp = MAE().calculate(target_data, y_pred_comp)


print(f"MSE аналитической модели: {mse_anal:.4f}")
print(f"MAE градиентной модели: {mse_comp:.4f}")

MSE аналитической модели: 1.0592
MAE градиентной модели: 0.7215


## Графики

In [389]:
fig = go.Figure()


fig.add_trace(
    go.Scatter(
        x= df_cut['square'], 
        y= df_cut['totalPrice'], 
        mode='markers',
        name='Исходные данные',
    )
)

fig.add_trace(
    go.Scatter(
        x=df_cut['square'], 
        y=y_pred_anal, 
        mode='lines',
        name='Предсказания аналитической модели',
    )
)

fig.add_trace(
    go.Scatter(
        x=df_cut['square'], 
        y=y_pred_comp, 
        mode='lines',
        name='Предсказания градиентной модели',
    )
)

fig.update_layout(
    title='Линейная регрессия: площадь vs цена',
    xaxis_title='Площать', 
    yaxis_title='Цена (нормализованная)',
    
    # Дополнительная настройка осей (опционально)
    xaxis=dict(showgrid=True, zeroline=True),
    yaxis=dict(showgrid=True, zeroline=True)
)


fig.show()


# Plotly приколы

In [ ]:
data = {
    "Category": ["A", "B", "C", "D"],
    "Value": [10, 25, 15, 30]
}
df = pd.DataFrame(data)
fig = px.bar(df, x="Category", y="Value", title="Sample Bar Chart")
fig.show()